# Construcción stop times
TODO: velocidad para recorrer entre segmento por zona geográfica

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import numpy as np
from pyproj import CRS  # <--- Esta es la línea que falta



## Parámetros

In [2]:
import json
from pathlib import Path as PathLib

_params_path = PathLib.cwd() / "params.json"
if not _params_path.exists():
    _params_path = PathLib("params.json")
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

CIUDAD = p["ciudad"]
dwell_time_station_minutes = p["stop_times"]["dwell_time_station_minutes"]
velocidad_kmh = p["stop_times"]["velocidad_kmh"]

In [3]:
# dwell_time_station_minutes y velocidad_kmh cargados desde params.json en la celda anterior


rutas_entrada

In [4]:
# --- Carpeta routes.txt ---
OUTPUT_DIR_gtfs = Path(f"../data/{CIUDAD}/gtfs-output")
OUTPUT_DIR_gtfs.mkdir(parents=True, exist_ok=True)
print(f"Entrada: {OUTPUT_DIR_gtfs.absolute()}")

# --- Carpeta processed routes ---
OUTPUT_DIR_processed = Path(f"../data/{CIUDAD}/processed")
OUTPUT_DIR_processed.mkdir(parents=True, exist_ok=True)
print(f"Entrada: {OUTPUT_DIR_processed.absolute()}")


Entrada: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/gtfs-output
Entrada: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/processed


rutas salida

In [5]:
# --- Carpeta routes.txt ---
OUTPUT_DIR_processed = Path(f"../data/{CIUDAD}/processed")
OUTPUT_DIR_processed.mkdir(parents=True, exist_ok=True)
print(f"Salida: {OUTPUT_DIR_processed.absolute()}")


Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/processed


## Lectura archivos

In [6]:
segments_gdf = gpd.read_file(OUTPUT_DIR_processed / "segments.geojson")
#stops_gtfs = pd.read_csv(OUTPUT_DIR_gtfs / "stops.txt")

## Calcular tiempo en recorrer cada segmento

In [7]:
segments_gdf.head()

,route_id,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,geometry
0,Route_1,shape_Route_1,0,Seg_Route_1_0000,Route_1_0000,Route_1_0001,0.0,200.0,"LINESTRING (-97.85418 22.21563, -97.85504 22.2..."
1,Route_1,shape_Route_1,1,Seg_Route_1_0001,Route_1_0001,Route_1_0002,200.0,400.0,"LINESTRING (-97.85504 22.21401, -97.85586 22.2..."
2,Route_1,shape_Route_1,2,Seg_Route_1_0002,Route_1_0002,Route_1_0003,400.0,600.0,"LINESTRING (-97.85578 22.21244, -97.85486 22.2..."
3,Route_1,shape_Route_1,3,Seg_Route_1_0003,Route_1_0003,Route_1_0004,600.0,800.0,"LINESTRING (-97.85441 22.21309, -97.85354 22.2..."
4,Route_1,shape_Route_1,4,Seg_Route_1_0004,Route_1_0004,Route_1_0005,800.0,1000.0,"LINESTRING (-97.85354 22.21471, -97.85327 22.2..."


## Calcular tiempo en recorrer cada segmento

In [8]:

def _utm_from_centroid(gdf: gpd.GeoDataFrame) -> CRS:
    g4326 = gdf.to_crs(4326)
    c = g4326.union_all().centroid
    lon, lat = float(c.x), float(c.y)
    zone = int((lon + 180) // 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return CRS.from_epsg(epsg)

def _crs_is_metric(crs: CRS) -> bool:
    try:
        return any(ai.unit_name.lower().startswith(("metre", "meter")) for ai in crs.axis_info)
    except Exception:
        return False

In [9]:


def add_time_kmh_min(
    segments_gdf: gpd.GeoDataFrame,
    speed_kmh,                 # float | dict | pd.Series (por ruta) | pd.Series (por fila)
    by: str = "route_id",      # columna para mapear velocidad cuando sea dict/Series por ruta
    length_col: str = "length_m",
    inplace: bool = False
) -> gpd.GeoDataFrame:
    """
    Añade SOLO:
      - speed_kmh  (km/h)
      - time_min   (minutos)
    """
    if "geometry" not in segments_gdf.columns:
        raise ValueError("segments_gdf debe tener 'geometry'.")

    g = segments_gdf if inplace else segments_gdf.copy()

    # 1) Longitud en metros (si no existe, calcularla)
    if length_col in g.columns:
        len_m = g[length_col].to_numpy(dtype=float)
    else:
        if g.crs is None:
            raise ValueError("El GeoDataFrame no tiene CRS (ej. EPSG:4326).")
        crs_in = CRS.from_user_input(g.crs)
        crs_m = g.crs if _crs_is_metric(crs_in) else _utm_from_centroid(g)
        len_m = g.to_crs(crs_m).geometry.length.to_numpy()
        g[length_col] = len_m

    # 2) Resolver velocidad (km/h) vectorizada
    if np.isscalar(speed_kmh):
        v_kmh = np.full(len(g), float(speed_kmh), dtype=float)
    elif isinstance(speed_kmh, dict):
        v_kmh = pd.Series(speed_kmh).reindex(g[by]).to_numpy(dtype=float)
    elif isinstance(speed_kmh, pd.Series):
        v_kmh = (speed_kmh.to_numpy(dtype=float) if speed_kmh.index.equals(g.index)
                 else speed_kmh.reindex(g[by]).to_numpy(dtype=float))
    else:
        raise TypeError("speed_kmh debe ser float, dict o pd.Series")

    if np.any(~np.isfinite(v_kmh)) or np.any(v_kmh <= 0):
        raise ValueError("Velocidades km/h inválidas (faltantes o <= 0).")

    # 3) Tiempo en minutos (sin columnas intermedias)
    # time_min = (dist_km / kmh) * 60 = (len_m/1000) * 60 / v_kmh
    g["speed_kmh"] = v_kmh
    g["time_min_travel"]  = (g[length_col].to_numpy(dtype=float) / 1000.0) * 60.0 / v_kmh
    return g

In [10]:
# 1) Misma velocidad para todos (22 km/h)
#segments_con_tiempo = add_travel_time_kmh(segments_gdf, speed_kmh=22)

# 2) Velocidad por ruta (dict)
#vel_por_ruta = {"r74": 18, "r76": 22, "r81": 20}
#segments_con_tiempo = add_travel_time_kmh(segments_gdf, speed_kmh=vel_por_ruta)

# 3) Velocidad por fila (columna existente)
#segments_con_tiempo = add_travel_time_kmh(segments_gdf, speed_kmh=segments_gdf["v_kmh"])

In [11]:
segments_con_tiempo = add_time_kmh_min(segments_gdf, speed_kmh=velocidad_kmh)

segments_con_tiempo["dwell_time"] = dwell_time_station_minutes

segments_con_tiempo["total_time"] = segments_con_tiempo["time_min_travel"]  + segments_con_tiempo["dwell_time"] 
segments_con_tiempo.head(5)

/Users/danielbustillos/miniconda3/envs/analisis-general/lib/python3.10/site-packages/shapely/set_operations.py:421: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


,route_id,shape_id,segment_seq,segment_id,from_stop_id,to_stop_id,from_measure_m,to_measure_m,geometry,length_m,speed_kmh,time_min_travel,dwell_time,total_time
0,Route_1,shape_Route_1,0,Seg_Route_1_0000,Route_1_0000,Route_1_0001,0.0,200.0,"LINESTRING (-97.85418 22.21563, -97.85504 22.2...",200.0,28.13,0.426591,0.2,0.626591
1,Route_1,shape_Route_1,1,Seg_Route_1_0001,Route_1_0001,Route_1_0002,200.0,400.0,"LINESTRING (-97.85504 22.21401, -97.85586 22.2...",200.0,28.13,0.426591,0.2,0.626591
2,Route_1,shape_Route_1,2,Seg_Route_1_0002,Route_1_0002,Route_1_0003,400.0,600.0,"LINESTRING (-97.85578 22.21244, -97.85486 22.2...",200.0,28.13,0.426591,0.2,0.626591
3,Route_1,shape_Route_1,3,Seg_Route_1_0003,Route_1_0003,Route_1_0004,600.0,800.0,"LINESTRING (-97.85441 22.21309, -97.85354 22.2...",200.0,28.13,0.426591,0.2,0.626591
4,Route_1,shape_Route_1,4,Seg_Route_1_0004,Route_1_0004,Route_1_0005,800.0,1000.0,"LINESTRING (-97.85354 22.21471, -97.85327 22.2...",200.0,28.13,0.426591,0.2,0.626591


## Contruir stop times

In [12]:
def sec_to_gtfs_time(sec):
    """Convierte segundos desde medianoche a formato HH:MM:SS."""
    h = int(sec) // 3600
    m = (int(sec) % 3600) // 60
    s = int(sec) % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

dwell_sec = dwell_time_station_minutes * 60

filas = []
for (route_id, shape_id), grp in segments_con_tiempo.sort_values("segment_seq").groupby(["route_id", "shape_id"]):
    grp = grp.sort_values("segment_seq").reset_index(drop=True)
    trip_id = f"{route_id}_trip_00"
    tiempo_actual_sec = 0.0

    # Primera parada (origen del primer segmento)
    stop_id_ini = grp.iloc[0]["from_stop_id"]
    filas.append({
        "trip_id": trip_id,
        #"route_id": route_id,
        "timepoint": 1,
        "stop_id": stop_id_ini,
        "stop_sequence": 1,
        "arrival_time": sec_to_gtfs_time(0),
        "departure_time": sec_to_gtfs_time(dwell_sec),
    })
    tiempo_actual_sec = dwell_sec
    seq = 2

    # Resto de paradas (to_stop_id de cada segmento)
    for i, row in grp.iterrows():
        tiempo_actual_sec += row["time_min_travel"] * 60
        arrival = sec_to_gtfs_time(tiempo_actual_sec)
        tiempo_actual_sec += dwell_sec
        departure = sec_to_gtfs_time(tiempo_actual_sec)
        filas.append({
            "trip_id": trip_id,
            #"route_id": route_id,
            "timepoint": 1,
            "stop_id": row["to_stop_id"],
            "stop_sequence": seq,
            "arrival_time": arrival,
            "departure_time": departure,
        })
        seq += 1

stop_times = pd.DataFrame(filas)


In [13]:
stop_times.head()

,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time
0,Route_1_trip_00,1,Route_1_0000,1,00:00:00,00:00:12
1,Route_1_trip_00,1,Route_1_0001,2,00:00:37,00:00:49
2,Route_1_trip_00,1,Route_1_0002,3,00:01:15,00:01:27
3,Route_1_trip_00,1,Route_1_0003,4,00:01:52,00:02:04
4,Route_1_trip_00,1,Route_1_0004,5,00:02:30,00:02:42


## Export

In [14]:
# archivos procesamiento


In [15]:
stop_times.to_csv(OUTPUT_DIR_gtfs / "stop_times.txt", index=False)